# LSTM Encoder-Decoder

## Load data

Set directory

In [1]:
import sys
import os

# Find the project root (Speciale_Kode)
current_dir = os.getcwd()
project_root = current_dir

while os.path.basename(project_root) != "Speciale_Kode":
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.append(project_root)

Load data

In [2]:
import pandas as pd
from pathlib import Path
from Modules.read_data import read_data

# ── Configuration ─────────────────────────────────────────────────────────────
PRICE_ZONE = "DK1"              # "DK1" or "DK2"
TRAIN_WINDOW = 2 * 8760         # 2 years of hourly data
VAL_START = "2024-01-01 00:00:00"
VAL_WINDOW = 8784
PREDICT_PERIOD = 4 * 168        # 4-week validation windows
STRIDE = 13 * 168               # stride between folds
POST_VALIDATION_EXCLUDE_HOURS = 168
INCLUDE_REMAINING_2024_DURING_TRAINING = True
INCLUDE_PRICE_HISTORY_AS_INPUT = True   # DKPrice_lag1 added as exog feature
INCLUDE_LAGS = False
USE_FORECASTED_HISTORY = True
# ──────────────────────────────────────────────────────────────────────────────

(
    DK1_train, DK1_test,
    DK2_train, DK2_test,
    DK1_train_weather, DK1_test_weather,
    DK2_train_weather, DK2_test_weather,
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test  = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test  = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

dataset_train = dataset_train.sort_values("Time").reset_index(drop=True)
dataset_test  = dataset_test.sort_values("Time").reset_index(drop=True)

val_start_ts     = pd.Timestamp(VAL_START)
year_2024_start  = pd.Timestamp("2024-01-01 00:00:00")
year_2025_start  = pd.Timestamp("2025-01-01 00:00:00")

df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)

history = dataset_train.loc[dataset_train["Time"] < val_start_ts].copy().iloc[-TRAIN_WINDOW:]
if len(history) < TRAIN_WINDOW:
    raise ValueError(f"Not enough history: need {TRAIN_WINDOW} rows before {VAL_START}.")

data_2024 = dataset_train.loc[
    (dataset_train["Time"] >= year_2024_start) & (dataset_train["Time"] < year_2025_start)
].copy()

# Build validation windows
validation_idx, validation_windows = [], []
window_start = val_start_ts
while (window_start + pd.Timedelta(hours=PREDICT_PERIOD)) <= year_2025_start:
    window_end = window_start + pd.Timedelta(hours=PREDICT_PERIOD)
    mask = (data_2024["Time"] >= window_start) & (data_2024["Time"] < window_end)
    if mask.any():
        validation_idx.extend(data_2024.index[mask].tolist())
        validation_windows.append((window_start, window_end))
    window_start = window_start + pd.Timedelta(hours=STRIDE)
validation_idx = sorted(set(validation_idx))

post_validation_exclusion_idx = []
for _, wend in validation_windows:
    excl_end = min(wend + pd.Timedelta(hours=POST_VALIDATION_EXCLUDE_HOURS), year_2025_start)
    excl_mask = (data_2024["Time"] >= wend) & (data_2024["Time"] < excl_end)
    if excl_mask.any():
        post_validation_exclusion_idx.extend(data_2024.index[excl_mask].tolist())
post_validation_exclusion_idx = sorted(set(post_validation_exclusion_idx))
excluded_from_remainder_idx = sorted(set(validation_idx).union(post_validation_exclusion_idx))

dataset_validation  = data_2024.loc[validation_idx].copy().sort_values("Time").reset_index(drop=True)
remainder_2024      = data_2024.drop(index=excluded_from_remainder_idx).copy().sort_values("Time").reset_index(drop=True)

if INCLUDE_REMAINING_2024_DURING_TRAINING:
    dataset_train = (
        pd.concat([history, remainder_2024], ignore_index=True)
        .sort_values("Time").drop_duplicates(subset=["Time"], keep="last").reset_index(drop=True)
    )
else:
    dataset_train = history.copy().sort_values("Time").reset_index(drop=True)

dataset_context = (
    pd.concat([history, data_2024], ignore_index=True)
    .sort_values("Time").drop_duplicates(subset=["Time"], keep="last").reset_index(drop=True)
)


dataset_train_full      = dataset_train.copy()
dataset_validation_full = dataset_validation.copy()
dataset_train_input     = dataset_train_full.copy()
dataset_validation_input = dataset_validation_full.copy()

if not INCLUDE_PRICE_HISTORY_AS_INPUT:
    dataset_train_input      = dataset_train_input.drop(columns=["DKPrice"])
    dataset_validation_input = dataset_validation_input.drop(columns=["DKPrice"])

lag_columns = [c for c in dataset_train_input.columns if '_lag' in c]
if not INCLUDE_LAGS:
    dataset_train_input      = dataset_train_input.drop(columns=lag_columns, errors='ignore')
    dataset_validation_input = dataset_validation_input.drop(
        columns=[c for c in dataset_validation_input.columns if '_lag' in c], errors='ignore'
    )

# Add DKPrice_lag1 as exog feature when price history is enabled
if INCLUDE_PRICE_HISTORY_AS_INPUT:
    price_lag_full = df[["Time", "DKPrice"]].copy().sort_values("Time").reset_index(drop=True)
    price_lag_full["DKPrice_lag1"] = price_lag_full["DKPrice"].shift(1)
    price_lag_full = price_lag_full[["Time", "DKPrice_lag1"]]
    dataset_train_input      = dataset_train_input.merge(price_lag_full, on="Time", how="left")
    dataset_validation_input = dataset_validation_input.merge(price_lag_full, on="Time", how="left")
    dataset_context = dataset_context.merge(price_lag_full, on="Time", how="left")

dataset_test = dataset_test.copy().reset_index(drop=True)

# Load precomputed feature forecasts
def _load_feature_predictions_for_zone(zone):
    prediction_path = Path(project_root) / "Data" / f"feature_predictions_{zone}_2024-2025.csv"
    if not prediction_path.exists():
        print("No precomputed forecasts found.")
        return None
    predictions = pd.read_csv(prediction_path, sep=";", decimal=".", parse_dates=["Time"], dayfirst=True)
    predictions = predictions.loc[:, ~predictions.columns.duplicated()].copy()
    if "DKZone" in predictions.columns:
        predictions = predictions.loc[predictions["DKZone"] == zone].copy()
        print(f"Loaded {len(predictions)} forecasts for zone {zone}.")
    return predictions

feature_predictions = _load_feature_predictions_for_zone(PRICE_ZONE)
use_precomputed_feature_values = feature_predictions is not None

print(f"Zone: {PRICE_ZONE}")
print(f"Train shape: {dataset_train.shape}")
print(f"Validation shape: {dataset_validation.shape}")
print(f"Test shape: {dataset_test.shape}")
print(f"Training input columns ({len(dataset_train_input.columns)}): {dataset_train_input.columns.tolist()}")
print(f"Validation windows: {len(validation_windows)}")
for i, (ws, we) in enumerate(validation_windows, 1):
    print(f"  {i:02d}. {ws} -> {we}")

Notebook_dir: c:\Users\chris\Documents\Python\Speciale_Kode\Modules
Python_dir: c:\Users\chris\Documents\Python\Speciale_Kode
Data_folder: c:\Users\chris\Documents\Python\Speciale_Kode\Data
Training data shape (DK1): (78888, 38)
Test data shape (DK1): (8760, 38)
Test set fraction (DK1): 9.99%
Training data shape (DK2): (78888, 38)
Test data shape (DK2): (8760, 38)
Test set fraction (DK2): 9.99%
Loaded 17544 forecasts for zone DK1.
Zone: DK1
Train shape: (22944, 38)
Validation shape: (2688, 38)
Test shape: (8760, 38)
Training input columns (35): ['DKPrice', 'Time', 'OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'Onsh

Load Random Forest forecasting models (if no precomputed features)

In [3]:
from Modules.Load_RF_forecast_models import load_rf_models

rf_models = None
if not use_precomputed_feature_values:
    rf_models = load_rf_models(user="Nikolaj")

Test CUDA

In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
Device: cuda
GPU: NVIDIA GeForce GTX 1660


## Prepare training data & scaler

In [5]:
import numpy as np
from sklearn.preprocessing import StandardScaler

target_col    = "DKPrice"
feature_cols  = [c for c in dataset_train_input.columns if c not in ["Time", target_col]]

# Drop rows with NaN in features/target before training
train_clean = dataset_train_input.dropna(subset=feature_cols + [target_col]).copy()

X_train = train_clean[feature_cols].values.astype(np.float32)
y_train = train_clean[target_col].values.astype(np.float32)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Training samples : {len(X_train)}")
print(f"Feature columns  : {len(feature_cols)}")
print(feature_cols)

Training samples : 22944
Feature columns  : 33
['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity', 'DKPrice_lag1']


## LSTM Encoder-Decoder

### Helper / Model definition

`lstm_encoder_decoder.py` lives in `Modules/` (same folder as `week_predictions2.py`).

In [6]:
from Modules.lstm_encoder_decoder import TorchLSTMEncoderDecoder

### Hyperparameter search

In [7]:
import numpy as np

enc_dec_param_grid = {
    "hidden_size"           : [64, 128],
    "layers"                : [1, 2],
    "learning_rate"         : [0.001, 0.0005],
    "max_epochs"            : [60],
    "patience"              : [10],
    "batch_size"            : [32, 64],
    "sequence_length"       : [24, 168],
    "dropout"               : [0.0, 0.2],
    "training_prediction"   : ["mixed_teacher_forcing"],
    "teacher_forcing_ratio" : [0.5],
    "dynamic_tf"            : [False],
}

total = np.prod([len(v) for v in enc_dec_param_grid.values()])
print(f"Total LSTM Enc-Dec combinations: {total}")

Total LSTM Enc-Dec combinations: 64


In [8]:
import itertools
import time
import joblib
import wandb
from Modules.Validation3 import run_cross_validation
from Modules.week_predictions2 import clear_forecast_feature_cache

WANDB_PROJECT_ENC_DEC = f"LSTM_EncDec_param_search_{PRICE_ZONE}"

keys   = list(enc_dec_param_grid.keys())
combos = list(itertools.product(*enc_dec_param_grid.values()))

# Filter: skip dropout > 0 when layers == 1 (no effect)
combos = [
    c for c in combos
    if not (dict(zip(keys, c))["layers"] == 1 and dict(zip(keys, c))["dropout"] > 0)
]
print(f"Actual combinations after filtering: {len(combos)}")

# ── Feature columns used for cross-validation ─────────────────────────────────
cv_feature_cols = [c for c in dataset_train_input.columns if c not in ["Time", target_col]]

# Prepare dataset_train_input with the target as first column for Validation3
dataset_train_cv = dataset_train_input[[target_col, "Time"] + cv_feature_cols].copy()
dataset_val_cv   = dataset_validation_input[[target_col, "Time"] + cv_feature_cols].copy()
dataset_ctx_cv   = dataset_context[[c for c in dataset_context.columns
                                    if c == "Time" or c == target_col or c in cv_feature_cols]].copy()

print(f"CV feature columns ({len(cv_feature_cols)}): {cv_feature_cols}")

start_time = time.time()
best_enc_dec_smape = float("inf")
best_enc_dec_model = None

for comb_no, combo in enumerate(combos, start=1):
    params = dict(zip(keys, combo))

    elapsed_min = (time.time() - start_time) / 60
    eta_min     = (elapsed_min / max(comb_no - 1, 1)) * len(combos) if comb_no > 1 else 0
    print(f"\nCombination {comb_no}/{len(combos)}: {params}")
    print(f"Time: {elapsed_min:.2f} min  - estimated total: {eta_min:.2f} min")

    run_name = (
        f"enc_dec_h{params['hidden_size']}_l{params['layers']}"
        f"_do{params['dropout']}_seq{params['sequence_length']}"
        f"_lr{params['learning_rate']}_bs{params['batch_size']}"
        f"_comb{comb_no:03d}"
    )

    wandb_run = wandb.init(
        project=WANDB_PROJECT_ENC_DEC,
        name=run_name,
        config=params,
        reinit=True,
        settings=wandb.Settings(start_method="thread"),
    )

    model = TorchLSTMEncoderDecoder(
        hidden_size           = params["hidden_size"],
        layers                = params["layers"],
        learning_rate         = params["learning_rate"],
        epochs                = params["max_epochs"],
        batch_size            = params["batch_size"],
        sequence_length       = params["sequence_length"],
        dropout               = params["dropout"],
        teacher_forcing_ratio = params["teacher_forcing_ratio"],
        training_prediction   = params["training_prediction"],
        dynamic_tf            = params["dynamic_tf"],
        patience              = params["patience"],
        log_epoch_metrics     = True,
        log_prefix            = "",
    )

    # Train epoch-by-epoch so we can validate after each epoch
    model.warm_start = True
    best_val_smape   = float("inf")
    patience_counter = 0

    for epoch in range(1, params["max_epochs"] + 1):
        print(f"  Epoch {epoch}/{params['max_epochs']}", end="\r")
        model.epochs = 1
        model.fit(X_train_scaled, y_train)

        t0 = time.time()
        clear_forecast_feature_cache()
        cv_results = run_cross_validation(
            model                        = model,
            dataset_train                = dataset_train_cv,
            dk_zone                      = PRICE_ZONE,
            split_setup                  = 2,
            train_window                 = TRAIN_WINDOW,
            val_window                   = VAL_WINDOW,
            val_start                    = VAL_START,
            predict_period               = PREDICT_PERIOD,
            stride                       = STRIDE,
            use_scaler                   = True,
            print_fold_results           = False,
            plot                         = False,
            rf_models                    = rf_models,
            use_precomputed_feature_values = use_precomputed_feature_values,
            precomputed_feature_predictions = feature_predictions,
            use_forecasted_history       = USE_FORECASTED_HISTORY,
            dataset_validation           = dataset_val_cv,
            include_remaining_2024       = INCLUDE_REMAINING_2024_DURING_TRAINING,
            dataset_context              = dataset_ctx_cv,
            feature_columns              = cv_feature_cols,
        )
        train_time = time.time() - t0

        val_smape = cv_results["overall_avg_weekly_smape"]
        print(f"Model trained in {train_time:.2f}s. Avg val SMAPE: {val_smape:.3f}")

        wandb.log({"epoch": epoch, "val_smape": val_smape})

        if val_smape < best_val_smape - 1e-4:
            best_val_smape   = val_smape
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= params["patience"]:
                print(f"  Early stopping at epoch {epoch}.")
                break

    wandb.summary["best_val_SMAPE"] = best_val_smape
    print(f"\n  best_val_SMAPE={best_val_smape:.3f}")

    if best_val_smape < best_enc_dec_smape:
        best_enc_dec_smape = best_val_smape
        best_enc_dec_model = model
        artifact = wandb.Artifact(f"enc_dec_{PRICE_ZONE}_best", type="model")
        with artifact.new_file("model.joblib", mode="wb") as f:
            joblib.dump(model, f)
        wandb_run.log_artifact(artifact)

    wandb.finish()

print(f"\nBest LSTM Enc-Dec SMAPE: {best_enc_dec_smape:.3f}")

c:\Users\chris\anaconda3\envs\ds809\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Actual combinations after filtering: 48
CV feature columns (33): ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity', 'DKPrice_lag1']

Combination 1/48: {'hidden_size': 64, 'layers': 1, 'learning_rate': 0.001, 'max_epochs': 60, 'patience': 10, 'batch_size': 32, 'sequence_length': 24, 'dropout': 0.0, 'training_prediction': 'mixed_teacher_forcing', 'teacher_forcing_ratio': 0.5, 'dynamic_tf': False}
Time: 0.00 min  - estimated total: 0.00 min


wandb: Currently logged in as: chrso19 (Energinet_speciale) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Model trained in 3.90s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 135.813
Model trained in 72.24s. Avg val SMAPE: 135.813
Model trained in 3.07s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 108.328
Model trained in 72.88s. Avg val SMAPE: 108.328
Model trained in 3.16s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 81.862
Model trained in 72.98s. Avg val SMAPE: 81.862
Model trained in 3.25s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 70.737
Model trained in 71.57s. Avg val SMAPE: 70.737
Model trained in 3.08s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 71.929
Model trained in 70.69s. Avg val SMAPE: 71.929
Model trained in 3.45s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 75.537
Model trained in 75.89s. Avg val SMAPE: 75.537
Model trained in 3.32s. Now validating on 4 folds...

Average SMAPE across all

epoch,▁▁▁▂▁▂▁▂▂▂▃▃▂▄▂▄▄▃▅▅▅▅▃▆▆▆▆▄▆▇▇▇▄▇▇██▄█▅
train_MSE_loss,██▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train_mae,██▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train_rmse,██▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train_smape,█▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_smape,█▅▂▁▁▂▂▂▁▁▁▁▂▂▁▁▁▁▁
best_val_SMAPE,70.23362
epoch,19
train_MSE_loss,346715.08065
train_mae,211.32359
train_rmse,588.82513



Combination 2/48: {'hidden_size': 64, 'layers': 1, 'learning_rate': 0.001, 'max_epochs': 60, 'patience': 10, 'batch_size': 32, 'sequence_length': 168, 'dropout': 0.0, 'training_prediction': 'mixed_teacher_forcing', 'teacher_forcing_ratio': 0.5, 'dynamic_tf': False}
Time: 24.45 min  - estimated total: 1173.40 min


Model trained in 3.76s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 143.391
Model trained in 69.09s. Avg val SMAPE: 143.391
Model trained in 3.67s. Now validating on 4 folds...


KeyboardInterrupt: 

## Final evaluation on 2025 test set

In [ ]:
from Modules.week_predictions2 import get_predictions
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_on_test(model, model_name: str, fitted_scaler):
    """Run get_predictions over the full 2025 test horizon and print metrics."""
    test_context = pd.concat(
        [dataset_context, dataset_test], ignore_index=True
    ).sort_values("Time").drop_duplicates(subset=["Time"], keep="last").reset_index(drop=True)

    # Keep only columns that match the CV feature set
    keep = [c for c in test_context.columns if c == "Time" or c == target_col or c in cv_feature_cols]
    test_context = test_context[keep].copy()

    test_preds = get_predictions(
        model                           = model,
        dataset                         = test_context,
        val_start                       = pd.Timestamp("2025-01-01 00:00:00"),
        val_end                         = dataset_test["Time"].max(),
        forecast_horizon                = 168,
        fitted_scaler                   = fitted_scaler,
        dk_zone                         = PRICE_ZONE,
        rf_models                       = rf_models,
        use_precomputed_feature_values  = use_precomputed_feature_values,
        precomputed_feature_predictions = feature_predictions,
        use_forecasted_history          = USE_FORECASTED_HISTORY,
    )

    pred_df = pd.concat(test_preds.values(), ignore_index=True)
    eval_df = pred_df.merge(dataset_test[["Time", target_col]], on="Time", how="inner").dropna()

    rmse  = np.sqrt(mean_squared_error(eval_df[target_col], eval_df["Prediction"]))
    mae   = mean_absolute_error(eval_df[target_col], eval_df["Prediction"])
    smape_val = smape_mean(eval_df[target_col].values, eval_df["Prediction"].values)

    print(f"\n{'='*50}")
    print(f"  {model_name} — 2025 Test Results")
    print(f"  RMSE  : {rmse:.3f}")
    print(f"  MAE   : {mae:.3f}")
    print(f"  SMAPE : {smape_val:.3f} %")
    print(f"{'='*50}")
    return eval_df

def smape_mean(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = np.abs(y_true) + np.abs(y_pred)
    vals   = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

if best_enc_dec_model is not None:
    enc_dec_test_df = evaluate_on_test(best_enc_dec_model, "LSTM Encoder-Decoder", scaler)